# Token cost explorer

What Claude has cost Birddog so far: spend by model, over time, per book and per page, and
how well the cost gate's estimates hold up. Data: `core.book_read` (one row per book, with
probe and assembly costs) and `core.book_page` (one row per page reading), written by the
`read_books` Airflow DAG (`birddog/books.py`).

Run all cells (**Run → Run All Cells**). Everything is read-only, as `birddog_reader`.
Prices are the list prices in `birddog/books.py` (`PRICES`); costs were computed per call when
the DAG ran, so they don't change if the prices are edited later.

In [ ]:
import os
import pandas as pd
import psycopg
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, HTML

DB_URL = os.environ["BIRDDOG_DATABASE_URL"]

# one fixed colour per model (validated categorical slots 1-3); never re-assigned by rank
MODEL_COLORS = {"claude-haiku-5-5": "#2a78d6", "claude-sonnet-5-5": "#eb6834", "claude-opus-5-5": "#1baf7a"}
MODEL_NAMES = {"claude-haiku-5-5": "Haiku 5.5", "claude-sonnet-5-5": "Sonnet 5.5", "claude-opus-5-5": "Opus 5.5"}
INK, INK_2, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e8e7e3"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "font.size": 10, "figure.dpi": 110, "legend.frameon": False,
})

def money(x, _=None):
    return f"${x:,.2f}" if abs(x) < 100 else f"${x:,.0f}"

def name(model):
    return MODEL_NAMES.get(model, model)

## Load

In [ ]:
with psycopg.connect(DB_URL, autocommit=True) as conn:
    books = pd.read_sql("""
        SELECT b.document_id, d.title, d.url, b.status, b.page_count, b.tokens_per_page,
               b.est_tokens, b.est_cost_usd, b.input_tokens, b.output_tokens,
               b.cost_usd::float AS cost_usd, b.run_id, b.queued_at, b.updated_at,
               b.places -> 'summary' AS summary
        FROM core.book_read b JOIN core.document d ON d.id = b.document_id""", conn)
    pages = pd.read_sql("""
        SELECT document_id, page_no, model, illegible, input_tokens, output_tokens,
               cost_usd::float AS cost_usd, created_at
        FROM core.book_page""", conn)
    vitals = pd.read_sql("""
        SELECT count(*) AS books, sum(page_count) AS pages FROM core.document
        WHERE 'Vitals' = ANY(doc_type) AND mimetype = 'application/pdf'""", conn)

# One usage ledger: every page reading, plus each book's probe + assembly (Haiku),
# which is the book's total minus its page readings.
page_cost = pages.groupby("document_id")[["cost_usd", "input_tokens", "output_tokens"]].sum()
other = books.set_index("document_id")[["cost_usd", "input_tokens", "output_tokens", "updated_at"]]
other = other.join(page_cost, rsuffix="_pages").fillna(0)
haiku = pd.DataFrame({
    "document_id": other.index,
    "model": "claude-haiku-5-5",
    "step": "probe + assemble",
    "cost_usd": (other["cost_usd"] - other["cost_usd_pages"]).clip(lower=0),
    "input_tokens": (other["input_tokens"] - other["input_tokens_pages"]).clip(lower=0),
    "output_tokens": (other["output_tokens"] - other["output_tokens_pages"]).clip(lower=0),
    "ts": other["updated_at"],
})
read = pages.assign(step=lambda p: p["model"].map(
    {"claude-sonnet-5-5": "read", "claude-opus-5-5": "re-read"}).fillna("read"))
read = read.rename(columns={"created_at": "ts"})[haiku.columns]
usage = pd.concat([read, haiku[haiku["cost_usd"] > 0]], ignore_index=True)
usage["ts"] = pd.to_datetime(usage["ts"], utc=True)
print(f"{len(books)} books, {len(pages)} page readings, {len(usage)} ledger rows")

## At a glance

In [ ]:
total = usage["cost_usd"].sum()
tok_in, tok_out = usage["input_tokens"].sum(), usage["output_tokens"].sum()
first_reads = pages[pages["model"] == "claude-sonnet-5-5"]
per_page = (first_reads["cost_usd"].sum() / len(first_reads)) if len(first_reads) else float("nan")
tok_page = ((first_reads["input_tokens"] + first_reads["output_tokens"]).sum() / len(first_reads)) if len(first_reads) else float("nan")
status = books["status"].value_counts()

def tile(label, value, note=""):
    return (f'<div style="padding:12px 16px;border:1px solid {GRID};border-radius:8px;min-width:150px">'
            f'<div style="color:{INK_2};font-size:12px">{label}</div>'
            f'<div style="color:{INK};font-size:26px;font-weight:600;font-variant-numeric:tabular-nums">{value}</div>'
            f'<div style="color:{INK_2};font-size:11px">{note}</div></div>')

display(HTML('<div style="display:flex;gap:12px;flex-wrap:wrap">' + "".join([
    tile("Total spend", money(total), f"{len(books)} books queued"),
    tile("Tokens", f"{(tok_in + tok_out) / 1e6:,.2f} M", f"{tok_in/1e6:,.2f} M in · {tok_out/1e6:,.2f} M out"),
    tile("Books read", f"{status.get('done', 0)}", f"{status.get('needs_approval', 0)} waiting for approval"),
    tile("Cost per page", money(per_page) if per_page == per_page else "–", "Sonnet first reading"),
    tile("Tokens per page", f"{tok_page:,.0f}" if tok_page == tok_page else "–", "image + prompt + answer"),
]) + "</div>"))
display(status.rename("books").to_frame())

## Spend by model

In [ ]:
by_model = (usage.groupby("model")
            .agg(cost_usd=("cost_usd", "sum"), input_tokens=("input_tokens", "sum"),
                 output_tokens=("output_tokens", "sum"))
            .reindex([m for m in MODEL_COLORS if m in set(usage["model"])]))
if by_model.empty:
    print("No spend yet.")
else:
    fig, ax = plt.subplots(figsize=(8, 0.6 + 0.55 * len(by_model)))
    labels = [name(m) for m in by_model.index]
    ax.barh(labels, by_model["cost_usd"], color=[MODEL_COLORS[m] for m in by_model.index],
            height=0.55, edgecolor=SURFACE, linewidth=2)
    for y, v in enumerate(by_model["cost_usd"]):
        ax.text(v, y, f"  {money(v)}  ({v / total:.0%})", va="center", color=INK, fontsize=10)
    ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(money))
    ax.set_title("Spend by model"); ax.set_xlim(0, by_model["cost_usd"].max() * 1.35)
    plt.show()
    display(by_model.assign(model=labels).set_index("model").style.format(
        {"cost_usd": "${:,.4f}", "input_tokens": "{:,.0f}", "output_tokens": "{:,.0f}"}))

## Spend over time

In [ ]:
if usage.empty:
    print("No spend yet.")
else:
    span = usage["ts"].max() - usage["ts"].min()
    freq, label = ("h", "hour") if span < pd.Timedelta(days=3) else ("D", "day")
    over = (usage.assign(bucket=usage["ts"].dt.floor(freq))
            .pivot_table(index="bucket", columns="model", values="cost_usd", aggfunc="sum", fill_value=0)
            .reindex(columns=[m for m in MODEL_COLORS if m in set(usage["model"])]))
    fig, ax = plt.subplots(figsize=(9, 3.6))
    bottom = pd.Series(0.0, index=over.index)
    width = pd.Timedelta(hours=0.8) if freq == "h" else pd.Timedelta(days=0.8)
    for m in over.columns:
        ax.bar(over.index, over[m], bottom=bottom, width=width, color=MODEL_COLORS[m],
               label=name(m), edgecolor=SURFACE, linewidth=2)
        bottom += over[m]
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(money))
    ax.set_title(f"Spend per {label}"); ax.grid(axis="x", visible=False)
    ax.legend(loc="upper left", ncols=len(over.columns))
    fig.autofmt_xdate(); plt.show()
    cum = over.sum(axis=1).cumsum()
    display(over.assign(total=over.sum(axis=1), cumulative=cum)
            .rename(columns=MODEL_NAMES).style.format("${:,.4f}"))

## Estimate vs actual, per book

The probe estimates each book from three sample pages; books over the auto limit wait for
approval on that estimate. Points on the diagonal were estimated exactly; above it, the book
cost more than estimated.

In [ ]:
done = books[books["status"].isin(["done", "read"]) & books["est_cost_usd"].notna()].copy()
if done.empty:
    print("No fully read books yet.")
else:
    done["est_cost_usd"] = done["est_cost_usd"].astype(float)
    done["ratio"] = done["cost_usd"] / done["est_cost_usd"]
    fig, ax = plt.subplots(figsize=(6, 5))
    hi = max(done["cost_usd"].max(), done["est_cost_usd"].max()) * 1.1
    ax.plot([0, hi], [0, hi], color=INK_2, linewidth=1, linestyle="--", label="estimate = actual")
    ax.scatter(done["est_cost_usd"], done["cost_usd"], s=64, color=MODEL_COLORS["claude-haiku-5-5"],
               edgecolor=SURFACE, linewidth=2, zorder=3, label="book")
    for _, r in done.nlargest(3, "cost_usd").iterrows():
        ax.annotate(f"  {r['page_count']} p.", (r["est_cost_usd"], r["cost_usd"]), fontsize=9, color=INK_2)
    ax.set_xlim(0, hi); ax.set_ylim(0, hi)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(money)); ax.yaxis.set_major_formatter(mticker.FuncFormatter(money))
    ax.set_xlabel("estimated (probe)"); ax.set_ylabel("actual"); ax.set_title("Estimate vs actual")
    ax.legend(loc="upper left"); plt.show()
    print(f"actual / estimate: median {done['ratio'].median():.2f}, range {done['ratio'].min():.2f}–{done['ratio'].max():.2f}")
    display(done[["document_id", "page_count", "est_cost_usd", "cost_usd", "ratio", "title"]]
            .sort_values("ratio", ascending=False)
            .style.format({"est_cost_usd": "${:,.2f}", "cost_usd": "${:,.2f}", "ratio": "{:.2f}"}))

## Cost per page

In [ ]:
if pages.empty:
    print("No pages read yet.")
else:
    pp = pages.assign(tokens=pages["input_tokens"] + pages["output_tokens"])
    models = [m for m in MODEL_COLORS if m in set(pp["model"])]
    fig, ax = plt.subplots(figsize=(8, 0.8 + 0.7 * len(models)))
    for y, m in enumerate(models):
        vals = pp.loc[pp["model"] == m, "cost_usd"]
        jitter = (pd.Series(range(len(vals))) % 7 - 3) * 0.04
        ax.scatter(vals, y + jitter.values, s=12, color=MODEL_COLORS[m], alpha=0.6, linewidth=0)
        ax.plot([vals.median()] * 2, [y - 0.3, y + 0.3], color=INK, linewidth=2)
        ax.text(vals.max(), y, f"  median {money(vals.median())}", va="center", fontsize=9, color=INK)
    ax.set_yticks(range(len(models)), [name(m) for m in models]); ax.invert_yaxis()
    ax.grid(axis="y", visible=False); ax.xaxis.set_major_formatter(mticker.FuncFormatter(money))
    ax.set_title("Cost per page reading (dots: pages, bar: median)")
    plt.show()
    summary = pp.groupby("model").agg(pages=("page_no", "size"), illegible=("illegible", "sum"),
                                      median_cost=("cost_usd", "median"), mean_tokens=("tokens", "mean"))
    summary.index = summary.index.map(name)
    display(summary.style.format({"median_cost": "${:,.4f}", "mean_tokens": "{:,.0f}"}))
    reread = pp.loc[pp["model"] == "claude-opus-5-5", "document_id"].size
    first = pp.loc[pp["model"] == "claude-sonnet-5-5", "document_id"].size
    if first:
        print(f"Opus re-read {reread} of {first} pages ({reread / first:.0%}).")

## What's waiting, and what everything would cost

Books over the auto limit wait in `views.books__needs_approval` (Mathesar). The projection
scales the measured cost per page (Sonnet first reading plus the measured Opus re-read share
and Haiku overhead) to every vital-records book.

In [ ]:
waiting = books[books["status"] == "needs_approval"]
print(f"Waiting for approval: {len(waiting)} books, {waiting['page_count'].sum():,} pages, "
      f"estimated {money(waiting['est_cost_usd'].astype(float).sum())}")
display(waiting[["document_id", "page_count", "est_tokens", "est_cost_usd", "title"]]
        .sort_values("est_cost_usd", ascending=False).head(25))

read_pages = len(pages[pages["model"] == "claude-sonnet-5-5"])
if read_pages:
    all_in = usage["cost_usd"].sum() / read_pages      # everything spent, per page read
    v = vitals.iloc[0]
    print(f"All-in cost per page read so far: {money(all_in)}")
    print(f"All {int(v['books']):,} vital-record books ({int(v['pages']):,} pages): about {money(all_in * v['pages'])}")
    print(f"With the Batch API (half price, results within 24 h): about {money(all_in * v['pages'] / 2)}")

## Explore

Pick a status and a date range to list books with their cost, estimate and cost per page.

In [ ]:
import ipywidgets as w

statuses = ["(all)"] + sorted(books["status"].unique().tolist())
status_pick = w.Dropdown(options=statuses, value="(all)", description="Status")
since = w.DatePicker(description="Updated from")
sort_pick = w.Dropdown(options=["cost_usd", "page_count", "est_cost_usd", "updated_at"], value="cost_usd", description="Sort by")
out = w.Output()

def refresh(*_):
    df = books.copy()
    if status_pick.value != "(all)":
        df = df[df["status"] == status_pick.value]
    if since.value:
        df = df[pd.to_datetime(df["updated_at"], utc=True).dt.date >= since.value]
    df["cost_per_page"] = df["cost_usd"] / df["page_count"].where(df["page_count"] > 0)
    with out:
        out.clear_output()
        print(f"{len(df)} books, {money(df['cost_usd'].sum())}")
        display(df.sort_values(sort_pick.value, ascending=False)[
            ["document_id", "status", "page_count", "est_cost_usd", "cost_usd", "cost_per_page", "updated_at", "title"]]
            .style.format({"est_cost_usd": "${:,.2f}", "cost_usd": "${:,.4f}", "cost_per_page": "${:,.4f}"}, na_rep="–"))

for widget in (status_pick, since, sort_pick):
    widget.observe(refresh, names="value")
display(w.HBox([status_pick, since, sort_pick]), out)
refresh()